# Supplementary Figure 9 : PSTHs for each cluster

**Author:** Parviz Ghaderi  
**Date:** 2025  

## Description
This notebook reproduces Figure 4a and 4b from the manuscript:  
_"Contextual gating-capability-scaled-vote response by the frontal cortex supports flexible decision-making"_

The analysis includes:
- **Figure 4a**: Heatmaps showing neural activity patterns across different experimental conditions, along with distributions of brain areas, cell types, cortical layers, and selectivity indices
- **Figure 4b**: Peristimulus time histograms (PSTHs) for each identified neural cluster

## Requirements
```bash
pip install numpy matplotlib scipy
```

## Instructions
1. Ensure the data file `Data_Clustering.pkl` is in the `../processed_data/` directory
2. Run all cells in order
3. Figures will be saved as PDFs in the current directory


## 1. Import Libraries and Set Parameters


In [1]:
# Import required libraries
import os
import pickle
import numpy as np
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.signal import convolve
from scipy.signal.windows import gaussian
import math

matplotlib.use('TkAgg')# Set matplotlib parameters for publication-quality figures
matplotlib.rcParams['pdf.fonttype'] = 42  # Embed fonts as editable text
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.linewidth'] = 0.5
matplotlib.rcParams['xtick.major.width'] = 0.5
matplotlib.rcParams['ytick.major.width'] = 0.5
import matplotlib.pyplot as plt

# Global parameters
BIN_SIZE = 0.01  # 10 ms bin size
GAUSS_WINDOW = 0.1  # 100 ms gaussian window
GAUSSIAN_SMOOTHING = 0  # 0 = no smoothing, 1 = apply smoothing
CMIN, CMAX = -0.2, 0.2  # Colormap limits for heatmaps


## 2. Define Custom Functions


In [2]:
def PG_Scale2Color(value, low_range, high_range, zero):
    """
    Assigns an RGB color to a value by mapping it onto a custom colormap.
    
    Color mapping:
    - Cyan for values below 'low_range'
    - Dark blue through black for negative values approaching 'zero'
    - Black at 'zero'
    - Black through dark red for positive values near 'zero'
    - Yellow for values above 'high_range'
    
    Parameters:
        value (float): The data value to color
        low_range (float): Lower threshold (most negative value)
        high_range (float): Upper threshold (most positive value)
        zero (float): The zero point dividing positive and negative
        
    Returns:
        tuple: RGB values (each between 0 and 1)
    """
    # Define the custom colormap
    colors = [
        (0 / 255, 230 / 255, 230 / 255),   # Cyan for very negative values
        (0.0, 0.1, 1),                     # Dark blue near zero (negative side)
        (0.0, 0.0, 0.0),                   # Black at zero
        (1, 0.1, 0.0),                     # Dark red near zero (positive side)
        (1.0, 1.0, 0.0),                   # Yellow for very positive values
    ]
    
    positions = [0.0, 0.3, 0.5, 0.7, 1.0]
    custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", list(zip(positions, colors)), N=256)
    
    # Normalize the value to the colormap
    if value <= low_range:
        norm_value = 0.0
    elif value >= high_range:
        norm_value = 1.0
    else:
        norm_value = (value - low_range) / (high_range - low_range)
    
    color = custom_cmap(norm_value)
    return color[:3]  # Return RGB only, no alpha


def create_custom_colormap(low_range, high_range, zero, n_colors=256):
    """
    Create a custom colormap using the PG_Scale2Color function.
    
    Parameters:
        low_range (float): Lower threshold
        high_range (float): Upper threshold
        zero (float): Zero point
        n_colors (int): Number of colors in the colormap
        
    Returns:
        tuple: (ListedColormap, Normalize) for use with matplotlib
    """
    colors = np.zeros((n_colors, 3))
    values = np.linspace(low_range, high_range, n_colors)
    for i, val in enumerate(values):
        colors[i, :] = PG_Scale2Color(val, low_range, high_range, zero)
    return mcolors.ListedColormap(colors), mcolors.Normalize(vmin=low_range, vmax=high_range)


def plot_heatmap(ax, psth, cluster_indices, time2plot, start_bin, end_bin, 
                 row_start, row_end, cmin, cmax):
    """
    Plot a heatmap of neural activity for a specific time window.
    
    Parameters:
        ax: Matplotlib axis object
        psth: Neural activity data (neurons x time bins)
        cluster_indices: Indices of neurons in the current cluster
        time2plot: Time vector for x-axis
        start_bin: Starting time bin
        end_bin: Ending time bin
        row_start: Starting row position for this cluster
        row_end: Ending row position for this cluster
        cmin: Minimum value for colormap
        cmax: Maximum value for colormap
    """
    time_segment = time2plot[0:300]
    cdata = psth[cluster_indices, start_bin:end_bin]
    
    # Create custom colormap
    colors = [
        (0 / 255, 230 / 255, 230 / 255),   # Cyan
        (0.0, 0.1, 1),                     # Dark blue
        (0.0, 0.0, 0.0),                   # Black
        (1, 0.1, 0.0),                     # Dark red
        (1.0, 1.0, 0.0),                   # Yellow
    ]
    positions = [0.0, 0.3, 0.5, 0.7, 1.0]
    custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", 
                                                           list(zip(positions, colors)), 
                                                           N=256)
    
    ax.imshow(cdata,
              origin="upper",
              aspect="auto",
              extent=[time_segment[0], time_segment[-1], row_end, row_start],
              vmin=cmin,
              vmax=cmax,
              cmap="coolwarm")
    
    ax.axhline(y=row_end, color="w", linewidth=1)
    ax.set_xlim(0, 3)
    ax.set_ylim(10294, 1)


## 3. Load and Preprocess Data


In [5]:
# Load data from pickle file
current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
pkl_file_path = os.path.join(parent_directory, "processed_data", "Data_Clustering.pkl")

print(f"Loading data from: {pkl_file_path}")
with open(pkl_file_path, "rb") as f:
    Cl = pickle.load(f)



print("Data loaded successfully!")

# Extract data arrays
psth = Cl["Neural_Data_normalized_Ordered"]
area = Cl["Areas_Ordered"]
celltype = Cl["Type_Ordered"] 
depth = Cl["Depth_Ordered"]
layer = Cl["Layer_Ordered"]
ccf_location = Cl["CCF_location_Ordered"]
ccf_xyz = Cl["CCF_xyz_Ordered"]
SI = Cl["Selectivity_index"]
cluster_labels_ordered = Cl["Cluster_Counter_Ordered"]

# Apply optional Gaussian smoothing
if GAUSSIAN_SMOOTHING == 1:
    N = int(GAUSS_WINDOW / BIN_SIZE)
    if N > 1:
        g = gaussian(N, std=(N / 6.0))
        w = g / np.trapz(g)  # Normalize area = 1
        psth = np.array([convolve(row, w, mode='same') for row in psth])
        print("Applied Gaussian smoothing")

# Define cluster ordering (based on peak response)
cluster_ordered = [1, 24, 23, 2, 16, 5, 14, 11, 21, 7, 9, 29, 20, 28, 25, 17, 13, 22, 6, 26, 27, 10, 3, 19, 15, 12, 18, 8, 4]
unique_clusters = np.unique(cluster_labels_ordered)

# Create time vector
time2plot = np.arange(1, int(15*(1/BIN_SIZE))) / (1/BIN_SIZE)

print(f"Number of neurons: {psth.shape[0]}")
print(f"Number of time bins: {psth.shape[1]}")
print(f"Number of clusters: {len(unique_clusters)}")


Loading data from: d:\Ghaderi_Zenodo_data_code_Final\processed_data\Data_Clustering.pkl
Data loaded successfully!
Number of neurons: 10294
Number of time bins: 1500
Number of clusters: 29


## 4. Plot Cluster PSTHs

This figure shows the peristimulus time histograms (PSTHs) for each cluster, with all 5 experimental conditions overlaid.


In [8]:
# Convert PSTH to firing rate (Hz)
psth_fr = Cl["Neural_Data_normalized_Ordered"] / BIN_SIZE

# Define color codes for 5 conditions
colorcodes = np.array([
    [0,   0,   1  ],  # Blue - Condition 1
    [0, 0.5,   1  ],  # Light blue - Condition 2
    [1,   0,   0  ],  # Red - Condition 3
    [1, 0.5,   0  ],  # Orange - Condition 4
    [0,   0,   0  ]   # Black - Condition 5
])

# Calculate grid dimensions
n_clusters = len(unique_clusters)
n_cols = 6
n_rows = math.ceil(n_clusters / n_cols)

# Create figure
fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(20/2.54, 25/2.54), 
                        sharex=True, sharey=True)
axes = axes.flatten()
fig.subplots_adjust(wspace=0.05, hspace=0.05)

# Plot PSTH for each cluster
cluster_cnt = 0
for c_id in cluster_ordered:
    if cluster_cnt >= len(axes):
        break
    
    # Get neurons in this cluster
    rows_for_cluster = psth_fr[cluster_labels_ordered == c_id, :]
    if rows_for_cluster.size == 0:
        continue
    
    # Calculate mean and SEM
    mean_sig = np.nanmean(rows_for_cluster, axis=0)
    sem_sig = np.nanstd(rows_for_cluster, axis=0) / np.sqrt(rows_for_cluster.shape[0])
    curve_upper, curve_lower = mean_sig + sem_sig, mean_sig - sem_sig
    
    ax = axes[cluster_cnt]
    seg_len = int(3 * (1 / BIN_SIZE))  # Length of each segment (300 bins)
    x_segment = time2plot[:seg_len]  # Use the same x range for all conditions
    
    # Plot each condition
    for i_cond in range(5):
        st, en = i_cond * seg_len, (i_cond + 1) * seg_len
        y_mean = mean_sig[st:en]
        y_upper = curve_upper[st:en]
        y_lower = curve_lower[st:en]
        
        # Plot mean ± SEM
        ax.fill_between(x_segment, y_lower, y_upper, color=colorcodes[i_cond], 
                       alpha=0.2, linewidth=0)
        ax.plot(x_segment, y_mean, color=colorcodes[i_cond], linewidth=0.8, 
               label=f"Condition {i_cond + 1}")
    
    # Add vertical lines at stimulus onset and offset
    ax.axvline(x=1, color='k', linestyle='--', linewidth=1)
    ax.axvline(x=2, color='k', linestyle='--', linewidth=1)
    
    # Format axes
    ax.set_ylim([-30, 65])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.set_title(f"#{c_id}", fontsize=14)
    
    cluster_cnt += 1

# Add scale bar to the last populated axis
last_ax = axes[cluster_cnt - 1]
x_min, x_max = last_ax.get_xlim()
y_min, y_max = last_ax.get_ylim()

# Scale bar parameters
x_bar_length = 0.5   # 0.5 seconds
y_bar_length = 10     # 5 Hz

# Position scale bar
x_bar_start = x_max - x_bar_length - 0.1
y_bar_start = y_min + 50

# Draw scale bars
last_ax.plot([x_bar_start, x_bar_start + x_bar_length], 
             [y_bar_start, y_bar_start], color='k', linewidth=2)
last_ax.plot([x_bar_start, x_bar_start], 
             [y_bar_start, y_bar_start + y_bar_length], color='k', linewidth=2)

# Add scale bar labels
last_ax.text(x_bar_start + x_bar_length / 2, y_bar_start - 1, f'{x_bar_length}s', 
             ha='center', va='top', fontsize=10)
last_ax.text(x_bar_start - 0.08, y_bar_start + y_bar_length / 2, f'{y_bar_length} Hz', 
             ha='right', va='center', fontsize=10)

# Remove unused axes
for ax in axes[cluster_cnt:]:
    fig.delaxes(ax)

# Add legend
handles = [mpatches.Patch(color=colorcodes[i]) for i in range(len(colorcodes))]
labels = [f"Condition {i+1}" for i in range(len(colorcodes))]
fig.legend(handles=handles, labels=labels, loc='upper right', fontsize=6, 
          title="Conditions", bbox_to_anchor=(1.2, 1))

plt.tight_layout()
plt.show(block=False)
# Save figure
fig.savefig("../Supplementary_figures_pdf/Ghaderi2025_Sup_Figure9_clustersPSTH.pdf", format='pdf', dpi=300, bbox_inches='tight')

print("Figure 4 and Supplementary 9 saved")


Figure 4 and Supplementary 9 saved


## 6. Summary

This notebook successfully reproduced:

- **Supplementary Figure 9**: Detailed PSTHs for each cluster across experimental conditions

The analysis reveals distinct functional clusters with varying selectivity indices, anatomical distributions across cortical areas and layers, and diverse temporal response profiles.

---

## References

Ghaderi, P. et al. (2025). Contextual gating-capability-scaled-vote response by the frontal cortex supports flexible decision-making.
